# 10 — Prepare the Commission file manifest

Combine the four Commission sources at **one row per source file or unresolved source entry**. Every source is retained, including failed downloads, press releases, complaint rejections, and files flagged for text review. Source classifications are carried over; this notebook does not decide which records belong in a later analytical subset.

The first two columns are editable. `candidate_to_merge` starts with a unique chronological integer per row. In a later iteration, assign the same integer to rows you want merged. `candidate_to_delete` starts blank. Existing values in both columns are preserved when the notebook is rerun, matched by the stable `manifest_row_id`.

The output is `output/final_manifest/final_commission_file_manifest.xlsx`. Keep a copy before deliberately changing the source identity scheme.

## Paths and input files

Run from anywhere inside `eccjeu`. The default input paths are the outputs of notebooks 01–09. Adjust only `INPUTS` if your local filenames differ. Files are read as data; this notebook does not access the web or reprocess PDFs.

In [2]:
from pathlib import Path
from collections import Counter
import re
import pandas as pd

def project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in (p, *p.parents):
        if (candidate / "output").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Run inside the eccjeu repository (with data/ and output/), or set ROOT explicitly.")

ROOT = project_root()
INPUTS = {
    "legacy_download": ROOT / "output/com_legacy/com_legacy.xlsx",
    "legacy_processing": ROOT / "output/com_legacy/com_legacy_clean_file_manifest.xlsx",
    "db_download": ROOT / "output/com_dbcomp/db_comp_manifest.csv",
    "db_processing": ROOT / "output/com_dbcomp/db_comp_text_manifest.csv",
    "db_classification": ROOT / "output/com_dbcomp/db_comp_classification.csv",
    "registry_download": ROOT / "output/com_registry/com_registry.xlsx",
    "registry_processing": ROOT / "output/com_registry/com_registry_processing.xlsx",
    "manual_download": ROOT / "output/com_manual/com_manual_download_manifest.csv",
    "manual_processing": ROOT / "output/com_manual/com_manual_clean_file_manifest.csv",
}
OUTPUT = ROOT / "output/final_manifest/final_commission_file_manifest.xlsx"

# Existing files with alternative names can be selected here explicitly.
missing = [f"{name}: {path}" for name, path in INPUTS.items() if not path.is_file()]
if missing:
    raise FileNotFoundError("Input files missing. Update INPUTS above:\n" + "\n".join(missing))
print("Project:", ROOT)
print("Output:", OUTPUT)

Project: /home/edik/projects/ec_cjeu
Output: /home/edik/projects/ec_cjeu/output/final_manifest/final_commission_file_manifest.xlsx


## Common fields and parsing rules

Blank means unavailable, not zero. Dates are taken from source metadata; an ID-encoded date is used only where the processing file supplies no date. `source_role` (main/OJ/fallback) is kept separate from `document_class` (the kind of decision or publication). `case_numbers_raw` is preferred over manual numeric normalization.

In [3]:
def read(path, sheet=None):
    if path.suffix.lower() == ".xlsx":
        return pd.read_excel(path, sheet_name=sheet or 0, dtype=str, keep_default_na=False).fillna("")
    return pd.read_csv(path, dtype=str, keep_default_na=False).fillna("")

def val(row, key):
    return str(row.get(key, "") or "").strip()

def basename(path):
    return Path(path).name if path else ""

def date_value(s):
    s = str(s or "").strip()
    if not s:
        return pd.NaT
    dt = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    if pd.isna(dt):
        dt = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
    if pd.isna(dt):
        dt = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    return dt

def from_decision_ids(ids):
    match = re.search(r"COM_AT_\d+_(\d{4}-\d{2}-\d{2})", ids or "")
    return match.group(1) if match else ""

def decide_status(raw_status, proc_status):
    if raw_status not in ("downloaded", "already_present"):
        return "not_processed"
    return proc_status or "missing_processing_record"

def row(**kwargs):
    return kwargs

rows = []

## Legacy: documents, failed attempts, and entries without documents

The failed-attempt log contains alternatives to files that may have downloaded in another format. It remains visible as separate attempt rows and is not counted as a missing decision. The 129 source entries without a retained file are included.

In [4]:
lr = read(INPUTS["legacy_download"], "records")
ld = read(INPUTS["legacy_download"], "documents")
lm = read(INPUTS["legacy_download"], "missing_documents")
lf = read(INPUTS["legacy_download"], "download_failures")
lp = read(INPUTS["legacy_processing"], "documents")
assert ld.document_id.is_unique and lp.document_id.is_unique
assert set(ld.document_id) == set(lp.document_id), "Legacy document-processing handoff differs"
lrec = lr.set_index("source_entry_id").to_dict("index")
lproc = lp.set_index("document_id").to_dict("index")

for _, d in ld.iterrows():
    p = lproc[d.document_id]; meta = lrec[d.source_entry_id]
    rows.append(row(manifest_row_id="legacy:document:"+d.document_id,
        source="com_legacy", source_record_id=d.source_entry_id, source_file_id=d.document_id,
        decision_id="", date=val(meta,"decision_date"), case_numbers=val(meta,"case_numbers"),
        title=val(meta,"title"), parties="", celex=val(d,"celex"),
        document_class=val(meta,"decision_type"), source_role=val(d,"source_type"),
        source_url=val(d,"source_url"), downloaded_url=val(d,"source_url"),
        download_status="downloaded", processing_status=val(p,"processing_status"),
        raw_file_name=basename(val(p,"raw_path")), raw_path=val(p,"raw_path"),
        text_file_name=basename(val(p,"text_path")), text_path=val(p,"text_path"),
        file_format=val(d,"file_format"), extraction_method=val(p,"extraction_method"),
        ocr_used=val(p,"ocr_used"), page_count=val(p,"n_pages"), text_chars=val(p,"n_chars"),
        raw_sha256=val(p,"raw_sha256"), text_sha256=val(p,"text_sha256"),
        review_reason=val(p,"review_reason"), error=val(p,"error"),
        source_language=val(d,"language")))

# The same CELEX may have multiple failed format/language requests; each URL is an attempt.
for _, d in lf.iterrows():
    meta = lrec[d.source_entry_id]
    rows.append(row(manifest_row_id="legacy:failed:"+d.source_entry_id+":"+val(d,"source_url"),
        source="com_legacy", source_record_id=d.source_entry_id, source_file_id="",
        decision_id="", date=val(meta,"decision_date"), case_numbers=val(meta,"case_numbers"),
        title=val(meta,"title"), parties="", celex=val(d,"celex"),
        document_class=val(meta,"decision_type"), source_role="download_attempt",
        source_url=val(d,"source_url"), downloaded_url="", download_status="failed",
        processing_status="not_processed", raw_file_name="", raw_path="", text_file_name="",
        text_path="", file_format=val(d,"file_format"), extraction_method="", ocr_used="",
        page_count="", text_chars="", raw_sha256="", text_sha256="",
        review_reason="", error=val(d,"error"), source_language=val(d,"language")))

for _, d in lm.iterrows():
    meta = lrec[d.source_entry_id]
    rows.append(row(manifest_row_id="legacy:no_document:"+d.source_entry_id,
        source="com_legacy", source_record_id=d.source_entry_id, source_file_id="",
        decision_id="", date=val(meta,"decision_date"), case_numbers=val(meta,"case_numbers"),
        title=val(meta,"title"), parties="", celex=val(meta,"celex_ids"),
        document_class=val(meta,"decision_type"), source_role="source_entry_no_file",
        source_url=val(meta,"source_url"), downloaded_url="", download_status="no_document",
        processing_status="not_processed", raw_file_name="", raw_path="", text_file_name="",
        text_path="", file_format="", extraction_method="", ocr_used="", page_count="",
        text_chars="", raw_sha256="", text_sha256="", review_reason="", error="",
        source_language=""))
print("Legacy rows:",len(ld),"documents +",len(lf),"failed attempts +",len(lm),"entries without files")

Legacy rows: 1579 documents + 139 failed attempts + 129 entries without files


## DB-COMP: every file and its classifier label

`document_class` retains all six model classes, including press releases, summaries, and complaint rejections. `include` is copied to a separate field and does not remove rows.

In [5]:
bd = read(INPUTS["db_download"]); bp = read(INPUTS["db_processing"]); bc = read(INPUTS["db_classification"])
for frame in (bd,bp,bc):
    assert frame.file_url.is_unique, "DB-COMP file_url must identify each source row"
assert set(bd.file_url)==set(bp.file_url)==set(bc.file_url), "DB-COMP handoff differs"
bproc=bp.set_index("file_url").to_dict("index")
bclass=bc.set_index("file_url").to_dict("index")
for _, d in bd.iterrows():
    p=bproc[d.file_url]; c=bclass[d.file_url]
    downloaded=val(d,"downloaded").lower() in ("true","1","yes")
    raw=val(p,"raw_path") if downloaded else val(d,"local_path")
    text=val(p,"text_path") if downloaded else ""
    rows.append(row(manifest_row_id="dbcomp:url:"+d.file_url, source="com_dbcomp",
        source_record_id=val(d,"dbcomp_document_id"), source_file_id=val(d,"file_url"),
        decision_id="", date=val(d,"dbcomp_date"), case_numbers=val(d,"case_number"),
        title=val(d,"document_name"), parties="", celex="",
        document_class=val(c,"document_class"), source_role=val(d,"case_type"),
        source_url=val(d,"file_url"), downloaded_url=val(d,"file_url") if downloaded else "",
        download_status="downloaded" if downloaded else "failed",
        processing_status=decide_status("downloaded" if downloaded else "failed",val(p,"processing_status")),
        raw_file_name=val(d,"file_name"), raw_path=raw, text_file_name=basename(text), text_path=text,
        file_format="pdf", extraction_method=val(p,"extraction_method"), ocr_used=val(p,"ocr_used"),
        page_count=val(p,"n_pages"), text_chars=val(p,"n_chars"),
        raw_sha256=val(p,"raw_sha256"), text_sha256=val(p,"text_sha256"),
        review_reason=val(p,"fallback_reason"), error=val(d,"download_error") or val(p,"error"),
        source_language="", classifier_include=val(c,"include"),
        classification_reason=val(c,"classification_reason")))
print("DB-COMP rows:", len(bd),"— all model classes retained")

DB-COMP rows: 837 — all model classes retained


## Registry: download attempts and linked case metadata

`item_labels` describes the decision type and `source_roles` identifies main/OJ/fallback files. Parties are the registry's **case-level** `companies` field and may include entities beyond the specific document. Multiple decision IDs are retained as written; a date is taken from the decision table only for single-ID rows.

In [6]:
rd=read(INPUTS["registry_download"],"files")
rp=read(INPUTS["registry_processing"],"documents")
rc=read(INPUTS["registry_download"],"cases")
rdec=read(INPUTS["registry_download"],"decisions")
assert rd.file_id.is_unique and rp.file_id.is_unique
assert set(rp.file_id)==set(rd.loc[rd.status.eq("downloaded"),"file_id"]), "Registry downloaded/processed IDs differ"
rproc=rp.set_index("file_id").to_dict("index")
rcase=rc.drop_duplicates("case_number").set_index("case_number").to_dict("index")
rdecision=rdec.drop_duplicates("decision_id").set_index("decision_id").to_dict("index")
for _, d in rd.iterrows():
    p=rproc.get(d.file_id,{})
    caseno=val(d,"case_numbers")
    cm=rcase.get(caseno,{}) if ";" not in caseno else {}
    ids=val(d,"decision_ids"); dm=rdecision.get(ids,{}) if ";" not in ids else {}
    status=val(d,"status")
    raw=val(p,"raw_path") or val(d,"relative_path")
    text=val(p,"text_path")
    rows.append(row(manifest_row_id="registry:file:"+d.file_id, source="com_registry",
        source_record_id=val(d,"document_ids"), source_file_id=d.file_id,
        decision_id=ids, date=val(dm,"decision_date") or from_decision_ids(ids),
        case_numbers=caseno, title=val(cm,"case_title"), parties=val(cm,"companies"),
        celex="", document_class=val(d,"item_labels"), source_role=val(d,"source_roles"),
        source_url=val(d,"source_url"), downloaded_url=val(d,"selected_url"),
        download_status=status, processing_status=decide_status(status,val(p,"processing_status")),
        raw_file_name=basename(raw) if status=="downloaded" else "",
        raw_path=raw if status=="downloaded" else "",
        text_file_name=basename(text), text_path=text, file_format=val(d,"file_format"),
        extraction_method=val(p,"extraction_method"), ocr_used=val(p,"ocr_used"),
        page_count=val(p,"n_pages"), text_chars=val(p,"n_chars"),
        raw_sha256=val(p,"raw_sha256") or val(d,"sha256"), text_sha256=val(p,"text_sha256"),
        review_reason=val(p,"review_reason"), error=val(d,"error") or val(p,"error"),
        source_language=""))
print("Registry rows:",len(rd),"—",rd.status.value_counts().to_dict())

Registry rows: 1213 — {'downloaded': 1150, 'failed': 48, 'no_url_available': 15}


## Manually identified decisions

In [7]:
mdown=read(INPUTS["manual_download"]); mp=read(INPUTS["manual_processing"])
keys=["source_row","url_index_in_row"]
assert not mdown.duplicated(keys).any() and not mp.duplicated(keys).any()
merged=mdown.merge(mp,on=keys,how="left",suffixes=("_download","_processed"),validate="one_to_one",indicator=True)
for _, d in merged.iterrows():
    ok=val(d,"success").lower() in ("true","1","yes")
    raw=val(d,"raw_file_path_download")
    text=val(d,"clean_file_path")
    rows.append(row(manifest_row_id=f"manual:row:{d.source_row}:url:{d.url_index_in_row}",
        source="com_manual", source_record_id=val(d,"source_row"),
        source_file_id=f"{d.source_row}:{d.url_index_in_row}", decision_id="",
        date=val(d,"decision_date_download"), case_numbers=val(d,"case_numbers_raw_download"),
        title=val(d,"decision_title_download"), parties="", celex=val(d,"celex_download"),
        document_class=val(d,"decision_label_download"), source_role="manual_url",
        source_url=val(d,"source_url_download"), downloaded_url=val(d,"resolved_url_download"),
        download_status=val(d,"status"),
        processing_status=decide_status("downloaded" if ok else "failed",val(d,"processing_status")),
        raw_file_name=val(d,"file_name"), raw_path=raw if ok else "",
        text_file_name=val(d,"clean_file_name"), text_path=text,
        file_format=val(d,"file_format"), extraction_method=val(d,"extraction_method"),
        ocr_used=val(d,"ocr_used"), page_count=val(d,"n_pages"), text_chars=val(d,"n_chars"),
        raw_sha256=val(d,"sha256"), text_sha256=val(d,"clean_sha256"),
        review_reason=val(d,"review_reason"), error=val(d,"error_download") or val(d,"error_processed"),
        source_language=""))
print("Manual rows:",len(merged))

Manual rows: 16


## Chronological IDs, review flags, and export

Rows with no date sort last. Stable row IDs preserve edited candidate columns on rerun. New rows receive new, never-recycled integers; a later source update can therefore change the date order without renumbering your existing edits. The workbook has a single main sheet and a concise source summary.

In [8]:
columns=["manifest_row_id","source","source_record_id","source_file_id","decision_id",
 "date","case_numbers","title","parties","celex","document_class","source_role",
 "source_url","downloaded_url","download_status","processing_status","raw_file_name",
 "raw_path","text_file_name","text_path","file_format","extraction_method","ocr_used",
 "page_count","text_chars","raw_sha256","text_sha256","review_reason","error",
 "classifier_include","classification_reason"]
frame=pd.DataFrame(rows).reindex(columns=columns).fillna("")
assert frame.manifest_row_id.is_unique, "A source generated duplicate stable row IDs"
frame["date"]=frame["date"].map(date_value)
frame["page_count"]=pd.to_numeric(frame.page_count,errors="coerce").astype("Int64")
frame["text_chars"]=pd.to_numeric(frame.text_chars,errors="coerce").astype("Int64")
frame=frame.sort_values(["date","source","manifest_row_id"],na_position="last",kind="stable").reset_index(drop=True)

prior={}
if OUTPUT.exists():
    old=read(OUTPUT,"files")
    needed={"manifest_row_id","candidate_to_merge","candidate_to_delete"}
    if not needed.issubset(old.columns) or old.manifest_row_id.duplicated().any():
        raise ValueError("Existing output lacks unique stable IDs/candidate columns. Preserve it and resolve before overwriting.")
    prior=old.set_index("manifest_row_id")[["candidate_to_merge","candidate_to_delete"]].to_dict("index")
    existing=pd.to_numeric(old.candidate_to_merge,errors="coerce").dropna()
    next_id=int(existing.max())+1 if len(existing) else 1
else:
    next_id=1

merge_ids=[]; delete_ids=[]
for rid in frame.manifest_row_id:
    saved=prior.get(rid)
    if saved is None:
        merge_ids.append(next_id); delete_ids.append(""); next_id+=1
    else:
        if not str(saved["candidate_to_merge"]).strip():
            raise ValueError(f"Existing candidate_to_merge is blank for {rid}; fix it before rerunning")
        merge_ids.append(saved["candidate_to_merge"])
        delete_ids.append(saved["candidate_to_delete"])
frame.insert(0,"candidate_to_delete",delete_ids)
frame.insert(0,"candidate_to_merge",merge_ids)

# Technical flags are separate from the source's own processing_status.
def flag(d):
    if d.download_status not in ("downloaded","already_present"):
        return "no_source_file" if d.download_status in ("no_document","no_url_available") else "download_failed"
    if d.processing_status not in ("success","review"):
        return "missing_processed_text"
    if d.processing_status=="review": return "processing_review"
    if d.source=="com_legacy" and d.file_format=="html" and d.text_chars==514:
        return "check_514_character_html"
    if d.source=="com_dbcomp" and d.text_chars < 1000:
        return "check_short_text"
    return ""
frame["quality_flag"]=frame.apply(flag,axis=1)
summary=(frame.groupby(["source","download_status","processing_status"],dropna=False)
         .size().rename("file_rows").reset_index())
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
tmp=OUTPUT.with_name(OUTPUT.stem+".pending.xlsx")
with pd.ExcelWriter(tmp,engine="openpyxl",datetime_format="yyyy-mm-dd") as writer:
    frame.to_excel(writer,sheet_name="files",index=False)
    summary.to_excel(writer,sheet_name="summary",index=False)
    ws=writer.sheets["files"]
    ws.freeze_panes="C2"
    ws.auto_filter.ref=ws.dimensions
    from openpyxl.styles import Font,PatternFill
    from openpyxl.utils import get_column_letter
    ws.row_dimensions[1].height=30
    for cell in ws[1]:
        cell.fill=PatternFill("solid",fgColor="19334D")
        cell.font=Font(color="FFFFFF",bold=True)
    widths={"A":21,"B":21,"C":43,"D":17,"E":28,"F":42,"G":18,"H":22,"I":43,"J":45,
            "K":18,"L":38,"M":22,"N":58,"O":58,"P":22,"Q":24,"R":32,"S":62,
            "T":32,"U":62,"V":14,"W":22,"X":12,"Y":14,"Z":16}
    for letter,width in widths.items(): ws.column_dimensions[letter].width=width
    for name in ("candidate_to_merge","candidate_to_delete"):
        col=frame.columns.get_loc(name)+1
        for rowno in range(2,len(frame)+2):
            ws.cell(rowno,col).fill=PatternFill("solid",fgColor="FFF2CC")
    sw=writer.sheets["summary"]
    sw.freeze_panes="A2";sw.auto_filter.ref=sw.dimensions
    for cell in sw[1]:
        cell.fill=PatternFill("solid",fgColor="19334D")
        cell.font=Font(color="FFFFFF",bold=True)
    for col,width in {"A":22,"B":24,"C":27,"D":16}.items():sw.column_dimensions[col].width=width
tmp.replace(OUTPUT)
print("Wrote",len(frame),"rows to",OUTPUT)
print(summary.to_string(index=False))
print("Quality flags:",frame.quality_flag.value_counts().to_dict())

Wrote 3913 rows to /home/edik/projects/ec_cjeu/output/final_manifest/final_commission_file_manifest.xlsx
      source  download_status processing_status  file_rows
  com_dbcomp       downloaded           success        837
  com_legacy       downloaded            review          2
  com_legacy       downloaded           success       1577
  com_legacy           failed     not_processed        139
  com_legacy      no_document     not_processed        129
  com_manual       downloaded           success         16
com_registry       downloaded            review          9
com_registry       downloaded           success       1141
com_registry           failed     not_processed         48
com_registry no_url_available     not_processed         15
Quality flags: {'': 3543, 'download_failed': 187, 'no_source_file': 144, 'check_514_character_html': 24, 'processing_review': 11, 'check_short_text': 4}


## Verification

The handoff assertions above check all recorded downloads against the processing manifests. They cannot confirm that the raw PDF/HTML or generated text physically exists on your machine; run this optional check locally after export if you want that additional validation.

In [9]:
def available(path):
    if not path: return False
    p=Path(path)
    return p.is_file() or (ROOT/p).is_file() or (ROOT.parent/p).is_file()

downloaded=frame[frame.download_status.isin(["downloaded","already_present"])]
bad_raw=downloaded.loc[~downloaded.raw_path.map(available),["manifest_row_id","raw_path"]]
processed=frame[frame.processing_status.isin(["success","review"])]
bad_text=processed.loc[~processed.text_path.map(available),["manifest_row_id","text_path"]]
print("Raw paths missing on this machine:",len(bad_raw))
print("Text paths missing on this machine:",len(bad_text))
from IPython.display import display
display(bad_raw.head(10),bad_text.head(10))

Raw paths missing on this machine: 0
Text paths missing on this machine: 0


,manifest_row_id,raw_path


,manifest_row_id,text_path
